##Objetivo: transformar os dados brutos em uma base analítica capaz de responder às perguntas do projeto. O processo foi dividido em três camadas: staging (limpeza e padronização), intermediate (integração das tabelas) e mart (base final para análises e dashboard).

In [0]:
%sql
-- staging da tabela de leads (Tabela: olist_marketing_qualified_leads_dataset)
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.stg_leads AS
SELECT
mql_id,
first_contact_date,
landing_page_id,
CASE
WHEN origin IS NULL OR origin = 'unknown' THEN 'nao_identificado'
ELSE origin
END AS origin
FROM workspace.portfolio_1_olist.olist_marketing_qualified_leads_dataset;

In [0]:
%sql
-- staging da tabela de negocios (Tabela: olist_closed_deals_dataset)
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.stg_deals AS
SELECT
mql_id,
won_date,
business_segment,
CASE
WHEN business_type IS NULL THEN 'nao_informado'
ELSE business_type
END AS business_type,
CASE
WHEN lead_type IS NULL THEN 'nao_informado'
ELSE lead_type
END AS lead_type,
CASE
WHEN lead_type LIKE 'online%' THEN 'online'
WHEN lead_type = 'industry' THEN 'industry'
WHEN lead_type = 'offline' THEN 'offline'
WHEN lead_type = 'other' THEN 'outro'
ELSE 'nao_informado'
END AS lead_type_grupo
FROM workspace.portfolio_1_olist.olist_closed_deals_dataset;

In [0]:
%sql
-- intermediate
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.int_funil AS
SELECT
l.mql_id,
l.origin,
l.first_contact_date,
date_trunc('month', l.first_contact_date) AS mes_contato,
d.won_date,
CASE WHEN d.won_date IS NOT NULL THEN 1 ELSE 0 END AS converteu,
CASE
WHEN d.won_date < l.first_contact_date THEN NULL
ELSE DATEDIFF(d.won_date, l.first_contact_date)
END AS dias_ate_fechar,
d.business_type,
d.lead_type,
d.lead_type_grupo,
d.business_segment
FROM workspace.portfolio_1_olist.stg_leads AS l
LEFT JOIN workspace.portfolio_1_olist.stg_deals AS d
ON l.mql_id = d.mql_id;

In [0]:
%sql
-- mart (pergunta 1 e 2)
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.mart_canais AS
SELECT
origin,
COUNT(*) AS total_leads,
SUM(converteu) AS negocios_fechados,
ROUND(SUM(converteu) * 100.0 / COUNT(*), 2) AS taxa_conversao_pct,
ROUND(PERCENTILE(dias_ate_fechar, 0.5), 1) AS mediana_dias_ate_fechar
FROM workspace.portfolio_1_olist.int_funil
GROUP BY origin
ORDER BY taxa_conversao_pct DESC;

In [0]:
%sql
-- mart (pergunta 3)
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.mart_coorte_mensal AS
SELECT
mes_contato,
COUNT(*) AS total_leads,
SUM(converteu) AS negocios_fechados,
ROUND(SUM(converteu) * 100.0 / COUNT(*), 2) AS taxa_conversao_pct,
ROUND(PERCENTILE(dias_ate_fechar, 0.5), 1) AS mediana_dias_ate_fechar
FROM workspace.portfolio_1_olist.int_funil
GROUP BY mes_contato
ORDER BY mes_contato;

In [0]:
%sql
-- mart (pergunta 4)
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.mart_perfil_mensal AS
SELECT
mes_contato,
business_type,
COUNT(*) AS negocios,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY mes_contato), 1) AS pct_no_mes
FROM workspace.portfolio_1_olist.int_funil
WHERE converteu = 1
GROUP BY mes_contato, business_type
ORDER BY mes_contato, negocios DESC;

In [0]:
%sql
-- mart (pergunta 5)
CREATE OR REPLACE VIEW workspace.portfolio_1_olist.mart_porte_tipo AS
SELECT
lead_type,
business_type,
COUNT(*) AS negocios,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY lead_type), 1) AS pct_no_porte
FROM workspace.portfolio_1_olist.int_funil
WHERE converteu = 1
GROUP BY lead_type, business_type
ORDER BY lead_type, negocios DESC;

## Observações: 

### camada staging
Limpeza e padronização de cada tabela isoladamente. No 'origin', 'unknown' e nulo foram unificados em 'nao_identificado', preservando as demais categorias, inclusive os 'other'. Em 'business_type' e 'lead_type', apenas o nulo virou 'nao_informado'. 

### camada intermediate
Junção dos leads (Tabela: olist_marketing_qualified_leads_dataset) aos negócios fechados (Tabela: olist_closed_deals_dataset) por LEFT JOIN, preservando os 8.000 leads para viabilizar o cálculo da conversão. Foram criadas as métricas 'converteu', 'dias_ate_fechar' (com o único caso de data invertida tratado como nulo) e 'mes_contato'.

### camada mart
Camada final, com um mart por grão de pergunta: mart_canais (por canal, perguntas 1 e 2), mart_coorte_mensal (por mês de captação, pergunta 3), mart_perfil_mensal (perfil por mês, pergunta 4) e mart_porte_tipo (porte por tipo, pergunta 5).